# Fine-tune SmolVLA (action expert) on `swagatk/pirobot_teleop` — Colab

**Before running:** `Runtime → Change runtime type → A100 GPU`, then `Runtime → Run all`.

What this notebook does:
1. Mounts Google Drive (checkpoints, logs and provenance go to `MyDrive/smolvla_pirobot/<RUN_NAME>/`).
2. Installs `lerobot[smolvla]==0.6.1` (the version this project's code is written against).
3. Downloads the dataset from the Hugging Face Hub at a **pinned commit** to the Colab local disk.
4. Fine-tunes `lerobot/smolvla_base` with the vision encoder frozen and only the **action expert** trained
   (`freeze_vision_encoder=true`, `train_expert_only=true`), batch size 32.
5. Saves a checkpoint every `SAVE_FREQ` steps. LeRobot marks the latest one with a `last` **symlink**, which
   Google Drive cannot store, so training writes to `/content` and a background thread copies each finished
   checkpoint to Drive (keeping the newest `KEEP_ON_DRIVE`).
6. **Disconnects the runtime** when training ends (success *or* failure) so no compute units are wasted.
   Stopping the cell by hand (interrupt) does *not* disconnect.

**Resuming:** if Colab drops the session, just `Run all` again with the same `RUN_NAME`. The newest checkpoint
on Drive is restored and training resumes from it (same dataset commit, same config).

Caveat: the auto-disconnect lives in the training cell. If an *earlier* cell fails (e.g. install), `Run all`
stops there and the runtime stays connected until Colab's idle timeout.

In [ ]:
# ---- Configuration -------------------------------------------------------
DATASET_REPO     = "swagatk/pirobot_teleop"
DATASET_REVISION = None        # None = current `main`; the resolved commit sha is recorded and reused on resume
RUN_NAME         = "smolvla_pirobot_bs32_seed1000"

BATCH_SIZE  = 32
STEPS       = 20_000
SAVE_FREQ   = 2_000            # checkpoint every N steps (the last step is always saved)
EVAL_SPLIT  = 0.1              # fraction of episodes held out for eval loss
EVAL_STEPS  = 1_000            # eval loss on held-out episodes every N steps
LOG_FREQ    = 100
NUM_WORKERS = 8                # video decoding runs on CPU; lower if the runtime has few cores
SEED        = 1000

KEEP_ON_DRIVE  = 3             # newest checkpoints kept on Drive (older ones are deleted from Drive)
SYNC_EVERY_S   = 60            # how often the background thread looks for a new checkpoint

AUTO_DISCONNECT       = True   # disconnect the runtime when training finishes
DISCONNECT_ON_FAILURE = True   # ...also when training crashes (the log is on Drive)

LEROBOT_VERSION = "0.6.1"
DRIVE_ROOT = "/content/drive/MyDrive/smolvla_pirobot"
LOCAL_ROOT = "/content"

LOCAL_RUN = f"{LOCAL_ROOT}/runs/{RUN_NAME}"              # lerobot --output_dir (local disk)
DRIVE_RUN = f"{DRIVE_ROOT}/{RUN_NAME}"                   # synced copy on Drive
DATA_DIR  = f"{LOCAL_ROOT}/data/{DATASET_REPO.replace('/', '__')}"

In [ ]:
# ---- GPU check -----------------------------------------------------------
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
if "A100" not in gpu.stdout:
    print("WARNING: not an A100. Batch 32 may not fit on a 16 GB GPU (e.g. T4); lower BATCH_SIZE if it OOMs.")

In [ ]:
# ---- Mount Google Drive --------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_RUN, exist_ok=True)
print("Drive run dir:", DRIVE_RUN)

In [ ]:
# ---- Install LeRobot with SmolVLA extras (pinned) ------------------------
!pip install -q "lerobot[smolvla]=={LEROBOT_VERSION}"
!lerobot-train --help > /dev/null && echo "lerobot-train OK"

In [ ]:
# ---- Hugging Face token (optional) ---------------------------------------
# Needed only if the dataset is private. Add a Colab secret named HF_TOKEN (key icon in the left bar).
import os
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:
    print("No HF_TOKEN secret available, continuing anonymously:", type(e).__name__)
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN set")

In [ ]:
# ---- Download the dataset at a pinned commit -----------------------------
import json
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download

info_path = Path(DRIVE_RUN) / "run_info.json"
run_info = json.loads(info_path.read_text()) if info_path.exists() else {}

rev = DATASET_REVISION
saved_rev = run_info.get("dataset_revision")
if saved_rev is not None:
    if rev is not None and rev != saved_rev:
        raise ValueError(f"RUN_NAME {RUN_NAME!r} was started on dataset commit {saved_rev}, "
                         f"but DATASET_REVISION={rev}. Use a new RUN_NAME for a different dataset version.")
    rev = saved_rev
if rev is None:
    rev = HfApi(token=HF_TOKEN).dataset_info(DATASET_REPO).sha
DATASET_SHA = rev
print("Dataset commit:", DATASET_SHA)

snapshot_download(repo_id=DATASET_REPO, repo_type="dataset", revision=DATASET_SHA,
                  local_dir=DATA_DIR, token=HF_TOKEN)

meta = json.loads((Path(DATA_DIR) / "meta" / "info.json").read_text())
print({k: meta[k] for k in ("codebase_version", "fps", "total_episodes", "total_frames", "robot_type")})
print({k: v.get("shape") for k, v in meta["features"].items()})

In [ ]:
# ---- Checkpoint sync / resume helpers ------------------------------------
# lerobot layout: <output_dir>/checkpoints/<zero-padded step>/{pretrained_model/, training_state/}
# plus a `last` symlink that lerobot points at a step dir only after that checkpoint is fully written.
import os, re, shutil
from pathlib import Path

_STEP_RE = re.compile(r"^\d+$")


def step_dirs(ckpt_root):
    """Checkpoint step directories under ckpt_root, oldest first (symlinks and .partial excluded)."""
    root = Path(ckpt_root)
    if not root.is_dir():
        return []
    dirs = [d for d in root.iterdir() if d.is_dir() and not d.is_symlink() and _STEP_RE.match(d.name)]
    return sorted(dirs, key=lambda d: int(d.name))


def is_complete(ckpt_dir):
    d = Path(ckpt_dir)
    return ((d / "pretrained_model" / "train_config.json").is_file()
            and (d / "pretrained_model" / "model.safetensors").is_file()
            and (d / "training_state" / "training_step.json").is_file())


def local_last(ckpt_root):
    """Step dir that lerobot's `last` symlink points to, or None."""
    link = Path(ckpt_root) / "last"
    if not link.is_symlink():
        return None
    target = (link.parent / os.readlink(link))
    return target if is_complete(target) else None


def prune(ckpt_root, keep, protect=None):
    """Delete all but the newest `keep` complete step dirs (never `protect`)."""
    dirs = [d for d in step_dirs(ckpt_root) if is_complete(d)]
    removed = []
    for d in dirs[:-keep] if keep > 0 else dirs:
        if protect is not None and d.resolve() == Path(protect).resolve():
            continue
        shutil.rmtree(d)
        removed.append(d.name)
    return removed


def sync_last_to_drive(local_ckpt_root, drive_ckpt_root, keep_on_drive, keep_local=2):
    """Copy the checkpoint `last` points to onto Drive (atomically via a .partial dir), then prune."""
    last = local_last(local_ckpt_root)
    if last is None:
        return None
    drive_root = Path(drive_ckpt_root)
    drive_root.mkdir(parents=True, exist_ok=True)
    dst = drive_root / last.name
    if not (dst.exists() and is_complete(dst)):
        tmp = drive_root / (last.name + ".partial")
        if tmp.exists():
            shutil.rmtree(tmp)
        shutil.copytree(last, tmp)
        if dst.exists():
            shutil.rmtree(dst)
        tmp.rename(dst)
        (drive_root / "LATEST").write_text(last.name + "\n")
        print(f"[sync] checkpoint {last.name} -> {dst}", flush=True)
    prune(drive_root, keep_on_drive, protect=dst)
    prune(local_ckpt_root, keep_local, protect=last)
    return dst


def restore_latest_from_drive(drive_ckpt_root, local_ckpt_root):
    """Copy the newest complete Drive checkpoint to local disk and point `last` at it. Returns step or None."""
    done = [d for d in step_dirs(drive_ckpt_root) if is_complete(d)]
    if not done:
        return None
    src = done[-1]
    local_root = Path(local_ckpt_root)
    local_root.mkdir(parents=True, exist_ok=True)
    dst = local_root / src.name
    if not is_complete(dst):
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    link = local_root / "last"
    if link.is_symlink() or link.exists():
        link.unlink()
    link.symlink_to(src.name)
    return int(src.name)

In [ ]:
# ---- Train, sync checkpoints to Drive, then disconnect -------------------
import datetime as dt, json, os, platform, shutil, subprocess, sys, threading, traceback
from pathlib import Path

local_ckpt = Path(LOCAL_RUN) / "checkpoints"
drive_ckpt = Path(DRIVE_RUN) / "checkpoints"
stamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = Path(DRIVE_RUN) / f"train_{stamp}.log"

# Decide: fresh run, or resume (from this session's local disk, else from Drive).
resume_step = None
if local_last(local_ckpt) is not None:
    resume_step = int(local_last(local_ckpt).name)
    print("Resuming from local checkpoint", resume_step)
else:
    resume_step = restore_latest_from_drive(drive_ckpt, local_ckpt)
    if resume_step is not None:
        print("Restored checkpoint", resume_step, "from Drive; resuming")
    elif Path(LOCAL_RUN).exists():
        print("Removing local run dir with no complete checkpoint:", LOCAL_RUN)
        shutil.rmtree(LOCAL_RUN)

if resume_step is not None:
    cmd = ["lerobot-train", "--resume=true",
           f"--config_path={local_ckpt / 'last' / 'pretrained_model' / 'train_config.json'}"]
else:
    cmd = [
        "lerobot-train",
        "--policy.path=lerobot/smolvla_base",
        "--policy.device=cuda",
        "--policy.use_amp=true",
        "--policy.freeze_vision_encoder=true",
        "--policy.train_expert_only=true",
        "--policy.push_to_hub=false",
        f"--dataset.repo_id={DATASET_REPO}",
        f"--dataset.root={DATA_DIR}",
        "--dataset.image_transforms.enable=true",
        f"--dataset.eval_split={EVAL_SPLIT}",
        f"--eval_steps={EVAL_STEPS}",
        f"--batch_size={BATCH_SIZE}",
        f"--steps={STEPS}",
        f"--save_freq={SAVE_FREQ}",
        f"--log_freq={LOG_FREQ}",
        f"--num_workers={NUM_WORKERS}",
        f"--seed={SEED}",
        f"--output_dir={LOCAL_RUN}",
        f"--job_name={RUN_NAME}",
        "--wandb.enable=false",
    ]

# Provenance: versions, GPU, dataset commit, exact command (appended per attempt).
def _ver(pkg):
    r = subprocess.run([sys.executable, "-c", f"import importlib.metadata as m; print(m.version('{pkg}'))"],
                       capture_output=True, text=True)
    return r.stdout.strip() or None

info_path = Path(DRIVE_RUN) / "run_info.json"
run_info = json.loads(info_path.read_text()) if info_path.exists() else {}
run_info.setdefault("run_name", RUN_NAME)
run_info.setdefault("dataset_repo", DATASET_REPO)
run_info.setdefault("dataset_revision", DATASET_SHA)
run_info.setdefault("config", dict(BATCH_SIZE=BATCH_SIZE, STEPS=STEPS, SAVE_FREQ=SAVE_FREQ, EVAL_SPLIT=EVAL_SPLIT,
                                   EVAL_STEPS=EVAL_STEPS, SEED=SEED, NUM_WORKERS=NUM_WORKERS,
                                   base_policy="lerobot/smolvla_base"))
attempt = {
    "start": dt.datetime.now().isoformat(timespec="seconds"),
    "mode": "resume" if resume_step is not None else "fresh",
    "resume_from_step": resume_step,
    "command": cmd,
    "log": log_path.name,
    "gpu": gpu.stdout.strip(),
    "python": platform.python_version(),
    "versions": {p: _ver(p) for p in ("lerobot", "torch", "transformers", "torchcodec", "huggingface_hub")},
}
run_info.setdefault("attempts", []).append(attempt)
info_path.write_text(json.dumps(run_info, indent=2))
(Path(DRIVE_RUN) / f"pip_freeze_{stamp}.txt").write_text(
    subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
print("Command:", " ".join(cmd))

# Background thread: copy each new `last` checkpoint to Drive.
stop = threading.Event()
def _sync_loop():
    while not stop.wait(SYNC_EVERY_S):
        try:
            sync_last_to_drive(local_ckpt, drive_ckpt, KEEP_ON_DRIVE)
        except Exception as e:  # never kill training because of a sync hiccup
            print("[sync] error:", repr(e), flush=True)
syncer = threading.Thread(target=_sync_loop, daemon=True)

already_done = resume_step is not None and resume_step >= STEPS
returncode, interrupted, error = None, False, None
try:
    if already_done:
        print(f"Checkpoint {resume_step} >= STEPS={STEPS}: training already finished, nothing to run.")
        returncode = 0
    else:
        syncer.start()
        env = dict(os.environ, PYTHONUNBUFFERED="1")
        with open(log_path, "a") as log:
            proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                    bufsize=1, env=env)
            try:
                for line in proc.stdout:
                    print(line, end="")
                    log.write(line)
                    log.flush()
                returncode = proc.wait()
            except KeyboardInterrupt:
                interrupted = True
                proc.terminate()
                proc.wait(timeout=120)
                returncode = proc.returncode
except Exception:
    error = traceback.format_exc()
    print(error)
finally:
    stop.set()
    if syncer.is_alive():
        syncer.join(timeout=600)
    try:
        final = sync_last_to_drive(local_ckpt, drive_ckpt, KEEP_ON_DRIVE)
        print("Newest checkpoint on Drive:", final)
    except Exception as e:
        print("[sync] final sync failed:", repr(e))
        final = None
    attempt.update(end=dt.datetime.now().isoformat(timespec="seconds"), returncode=returncode,
                   interrupted=interrupted, error=error, final_checkpoint=str(final) if final else None)
    info_path.write_text(json.dumps(run_info, indent=2))

    ok = returncode == 0 and error is None and not interrupted
    print("Training", "FINISHED" if ok else ("INTERRUPTED" if interrupted else f"FAILED (returncode={returncode})"))
    if AUTO_DISCONNECT and not interrupted and (ok or DISCONNECT_ON_FAILURE):
        print("Flushing Drive and disconnecting the runtime...")
        drive.flush_and_unmount()
        from google.colab import runtime
        runtime.unassign()

## After training

On Drive, `MyDrive/smolvla_pirobot/<RUN_NAME>/` contains:
- `checkpoints/<step>/pretrained_model/`: the policy to pass to `laptop/smolvla_server.py`
  (the newest step is named in `checkpoints/LATEST`)
- `train_*.log`: full training log, including `eval_loss` on held-out episodes every `EVAL_STEPS` steps
- `run_info.json`: dataset commit, config, exact command, GPU, package versions per attempt
- `pip_freeze_*.txt`: full environment

Pick the checkpoint by held-out eval loss **and** by real driving runs on the robot; a lower loss doesn't
guarantee better navigation.